In [1]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.optim import Adam
import sklearn
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.checkpoint import checkpoint
from sklearn.preprocessing import OneHotEncoder

import cv2
import os
os.environ['QT_QPA_PLATFORM'] = 'xcb'
import pandas as pd


without embedding

In [2]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv3d_1 = nn.Conv3d(3, 32, (3, 3, 3), stride=(1, 2, 2),padding=1)
        self.pool_1 = nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2))
        self.conv3d_2 = nn.Conv3d(32, 64, (3, 3, 3),padding=1)
        self.pool_2 = nn.MaxPool3d((2, 2, 2), stride=(2, 2, 2))
        self.conv3d_3a = nn.Conv3d(64, 128, (3, 3, 3),padding=1)
        self.conv3d_3b = nn.Conv3d(128, 128, (3, 3, 3),padding=1)
        self.pool_3 = nn.MaxPool3d((2, 2, 2), stride=(2, 2, 2))
        self.conv3d_4a = nn.Conv3d(128, 128, (3, 3, 3),padding=1)
        self.conv3d_4b = nn.Conv3d(128, 128, (3, 3, 3),padding=1)
        self.pool_4 = nn.MaxPool3d((2, 2, 2), stride=(2, 2, 2))
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        self.dropout = nn.Dropout(0.25)
        self.fc = nn.Linear(128, 8)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # print(f"Input shape: {x.shape}")
        x = self.pool_1(F.relu(self.conv3d_1(x)))
        # print(f"After conv3d_1 and pool_1: {x.shape}")
        x = self.pool_2(F.relu(self.conv3d_2(x)))
        # print(f"After conv3d_2 and pool_2: {x.shape}")
        x = F.relu(self.conv3d_3a(x))
        # print(f"After conv3d_3a: {x.shape}")
        x = self.pool_3(F.relu(self.conv3d_3b(x)))
        # print(f"After conv3d_3b and pool_3: {x.shape}")
        x = F.relu(self.conv3d_4a(x))
        # print(f"After conv3d_4a: {x.shape}")
        x = self.pool_4(F.relu(self.conv3d_4b(x)))
        # print(f"After conv3d_4b and pool_4: {x.shape}")
        x = F.normalize(x, p=2, dim=1)  # Apply L2 normalization
        x = x.view(x.size(0), -1, 128)  # Flatten layer
        # print(f"After flattening: {x.shape}")
        x, _ = self.lstm(x)
        # print(f"After LSTM: {x.shape}")
        x = self.dropout(x)
        x = self.fc(x[:, -1, :])  # Apply fully connected layer to the last time step
        x = self.softmax(x)
        # print(f"Output shape: {x.shape}")
        return x

model = Net()

with embedding

In [21]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Video processing branch (3D CNN)
        self.conv3d_1 = nn.Conv3d(3, 32, (3, 3, 3), stride=(1, 2, 2), padding=1)
        self.pool_1 = nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2))
        self.conv3d_2 = nn.Conv3d(32, 64, (3, 3, 3), padding=1)
        self.pool_2 = nn.MaxPool3d((2, 2, 2), stride=(2, 2, 2))
        self.conv3d_3a = nn.Conv3d(64, 128, (3, 3, 3), padding=1)
        self.conv3d_3b = nn.Conv3d(128, 128, (3, 3, 3), padding=1)
        self.pool_3 = nn.MaxPool3d((2, 2, 2), stride=(2, 2, 2))
        self.conv3d_4a = nn.Conv3d(128, 128, (3, 3, 3), padding=1)
        self.conv3d_4b = nn.Conv3d(128, 128, (3, 3, 3), padding=1)
        self.pool_4 = nn.MaxPool3d((2, 2, 2), stride=(2, 2, 2))
        
        # Hand embedding branch
        self.embedding_fc = nn.Linear(21 * 3, 64)  # Assuming 21 3D landmarks
        
        # Combined processing
        self.lstm = nn.LSTM(128 + 64, 128, batch_first=True)  # Increased input size
        self.dropout = nn.Dropout(0.25)
        self.fc = nn.Linear(128, 8)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, video_input, hand_embeddings):
        # Process video branch
        x = self.pool_1(F.relu(self.conv3d_1(video_input)))
        x = self.pool_2(F.relu(self.conv3d_2(x)))
        x = F.relu(self.conv3d_3a(x))
        x = self.pool_3(F.relu(self.conv3d_3b(x)))
        x = F.relu(self.conv3d_4a(x))
        x = self.pool_4(F.relu(self.conv3d_4b(x)))
        x = F.normalize(x, p=2, dim=1)
        x = x.view(x.size(0), -1, 128)

        # Process hand embeddings
        h = F.relu(self.embedding_fc(hand_embeddings))
        h = h.unsqueeze(1).repeat(1, x.size(1), 1)  # Match temporal dimension
        
        # Concatenate features
        combined = torch.cat([x, h], dim=2)
        
        # Process combined features
        lstm_out, _ = self.lstm(combined)
        x = self.dropout(lstm_out)
        x = self.fc(x[:, -1, :])
        x = self.softmax(x)
        
        return x

# Usage
model = Net()

In [3]:
def load_checkpoint(checkpoint_path, model, optimizer):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    accuracy = checkpoint['accuracy']
    print(f'Checkpoint loaded from {checkpoint_path}')
    return epoch, loss, accuracy

In [4]:

def preprocess_frame(frame):
    # Resize the frame to the required input size of the model
    frame = cv2.resize(frame, (240, 240))  # Example size, change as needed
    
    # Convert the frame to RGB (if needed)
    if frame.shape[2] == 1:  # If the frame has only one channel
        frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)
    elif frame.shape[2] == 4:  # If the frame has an alpha channel
        frame = cv2.cvtColor(frame, cv2.COLOR_BGRA2RGB)
    else:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Normalize the frame (if needed)
    frame = frame / 255.0
    
    # Convert the frame to a PyTorch tensor
    frame = torch.tensor(frame, dtype=torch.float32)
    
    # Add a batch dimension
    # frame = frame.unsqueeze(0)
    

    
    return frame

In [5]:
def Image_Processing(frame,hands,top_gesture):

    image = cv2.cvtColor(cv2.flip(frame,1),cv2.COLOR_BGR2RGB)
    image.flags.writeable = False

    results = hands.process(image)
    
    image.flags.writeable = True
    image = cv2.cvtColor(image,cv2.COLOR_RGB2BGR)

    fontScale = 2
    fontFace = cv2.FONT_HERSHEY_PLAIN
    fontColor = (0,255,0)
    fontThickness = 2

    cv2.putText(image,top_gesture,(0,30),fontFace,fontScale,fontColor,fontThickness,cv2.LINE_AA)


    return image,results

In [6]:

def load_labels(csv_path):
    df = pd.read_csv(csv_path)
    label_dict = pd.Series(df.label.values, index=df.label_id).to_dict()
    return label_dict

# Load labels
label_dict = load_labels('/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/unique_labels.csv')

In [7]:
def run_inference(model, input_tensor, label_dict):
    with torch.no_grad():
        outputs = model(input_tensor)
        predicted_id = torch.argmax(outputs, 1).item()
        predicted_label = label_dict[predicted_id]
        return predicted_label

In [8]:
# def run_inference(model, input_tensor):
#     with torch.no_grad():
#         outputs = model(input_tensor)
#         predicted = torch.argmax(outputs, 1)
#         return predicted.item()

In [9]:
import mediapipe as mp
import torch.optim as optim

# Load the checkpoint model
optimizer = optim.Adam(model.parameters(), lr=0.001)

checkpoint_path = '/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/checkpoints/checkpoint_epoch_50.pth'
epoch, loss, accuracy = load_checkpoint(checkpoint_path, model, optimizer)
model.eval()  # Set the model to evaluation mode

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(1)
top_gesture = None

frame_buffer = []

with mp_hands.Hands(static_image_mode=False, max_num_hands=4, min_detection_confidence=0.7, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            print("Ignoring empty camera frame.")
            break

        # Preprocess the frame
        preprocessed_frame = preprocess_frame(frame)
        frame_buffer.append(preprocessed_frame)

        # If we have collected 30 frames, run inference
        if len(frame_buffer) == 30:
            # Stack frames to create a 5D tensor (N, C, D, H, W)
            input_tensor = torch.stack(frame_buffer).permute(3,0,1,2).unsqueeze(0)
            # input_tensor = input_tensor.to('cuda' if torch.cuda.is_available() else 'cpu')  # Move the tensor to the appropriate device

            # Run inference
            top_gesture = run_inference(model, input_tensor,label_dict)

            # Clear the buffer
            frame_buffer = []

            # Process the frame with MediaPipe Hands
        image, results = Image_Processing(frame, hands, str(top_gesture))
        
        h, w, c = image.shape

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # G.Gesture_Action(top_gesture, results)
            cv2.imshow('Hand Tracking', image)
        else:
            cv2.imshow('Hand Tracking', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

2024-11-11 19:56:44.515101: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731335204.527156   23229 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731335204.530700   23229 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-11 19:56:44.544032: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/tmp/ipykernel_23229/1951106302.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current defaul

Checkpoint loaded from /mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/checkpoints/checkpoint_epoch_50.pth


I0000 00:00:1731335206.711793   23229 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1731335206.715636   23429 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.6-arch1.1), renderer: AMD Radeon Graphics (radeonsi, renoir, LLVM 18.1.8, DRM 3.54, 6.6.59-1-lts)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1731335206.732836   23410 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1731335206.748101   23415 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1731335209.479478   23417 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
